# 02 - Baselines B0, B1 y B2

B0 predice la proporción de estados ganados en `train`. B1 calcula una puntuación fija de recursos y aprende en `train` cómo convertirla en probabilidad. B2 usa regresión logística regularizada con las características del estado. Los escenarios A, B y C se procesan por separado; cada partida tiene el mismo peso total. Este cuaderno no usa `calib` ni `test`.

La primera celda prepara este runtime: clona el código y descarga las tablas procesadas si faltan.


In [ ]:
# Prepara el runtime de Colab y descarga las tablas procesadas si hacen falta.
from pathlib import Path
import gc, sys
REPO_DIR = Path('/content/repo')
if not REPO_DIR.exists():
    !git clone -b 04MainAIModelTrainingAndDevelopment https://github.com/Adriel23456/Applied-AI-Agent-Project.git /content/repo
else:
    !git -C /content/repo pull
%cd /content/repo
!pip install -q -r requirements.txt
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
# Evita que Python conserve una versión vieja del código después de actualizar el repo.
for nombre in [m for m in list(sys.modules) if m == 'ptcg_wp' or m.startswith('ptcg_wp.')]:
    del sys.modules[nombre]
DATA_DIR = Path('/content/datos')
from ptcg_wp.ingest import load_days
dias = load_days('configs/dias.yaml')
# Archivos mínimos para modelar: tablas de estados por día y asignación de particiones.
esperados = [DATA_DIR / f'estados_{dia}.parquet' for dia in dias] + [DATA_DIR / 'particiones.parquet']
if not all(ruta.exists() for ruta in esperados):
    from ptcg_wp.colab import setup_kaggle
    setup_kaggle()  # Lee el token guardado en Colab o lo solicita.
    !PYTHONPATH=src python -m ptcg_wp.store bajar --dir /content/datos
assert all(ruta.exists() for ruta in esperados), 'Faltan tablas: revisa la descarga anterior.'

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from ptcg_wp.baselines import (ScenarioStream, weights_by_game, b0_and_hp_unit,
                               b1_probability, fit_b1_mapping)
from ptcg_wp.features import feature_columns
from ptcg_wp.b2 import (C_VALUES, b2_group_cv, summarize_b2_cv, fit_b2_candidates_val)
import pyarrow.parquet as pq
SCENARIOS = ('A', 'B', 'C')
print('Entorno listo; tablas disponibles:', len(list(DATA_DIR.glob('*.parquet'))), '| escenarios:', SCENARIOS)


## Peso igual por partida

Una partida con 200 estados da peso `1/200` a cada estado; una con 20 da `1/20`. Cada partida suma 1 en total. Para una métrica por etapa, los pesos se recalculan dentro de esa etapa.

In [ ]:
# Calcula AUC, log loss y Brier comparando etiquetas reales con probabilidades predichas.
def metricas(y, p, game_ids):
    w = weights_by_game(game_ids)
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)  # Evita probabilidades exactamente 0 o 1.
    return {
        'auc': roc_auc_score(y, p, sample_weight=w),
        'log_loss': log_loss(y, p, sample_weight=w, labels=[0, 1]),
        'brier': brier_score_loss(y, p, sample_weight=w),
        'n_estados': len(y),
        'n_partidas': pd.Series(game_ids).nunique(),
    }

# Calcula reportes para todos los estados y también por etapa de la partida.
def evaluar(val, pred, escenario, modelo):
    filas = []
    grupos = [('global', np.ones(len(val), dtype=bool))]
    grupos += [(etapa, val['stage'].to_numpy() == etapa) for etapa in ('inicio', 'medio', 'final')]
    for etapa, mascara in grupos:
        parte = val.loc[mascara]
        if len(parte):
            filas.append({'escenario': escenario, 'modelo': modelo, 'etapa': etapa, **metricas(parte.y, pred[mascara], parte.game_id)})  # ** añade las métricas al reporte.
    return filas
print('Funciones de ponderación y métricas listas.')


## B0 - probabilidad a priori

B0 ignora el tablero. Calcula en train la fracción de partidas ganadas por quien actúa y predice esa probabilidad en todos los estados de val. Es el piso de comparación.

## B1 - heurística de recursos

**Propuesta concreta para revisar:** `score = (opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n`. El signo queda positivo cuando el estado favorece al jugador que actúa. `hp_unit` será el promedio de `active_max_hp` positivo observado en los estados de `train`, ponderando para que cada partida tenga el mismo peso. Así, una ventaja en PV se expresa en unidades del HP promedio observado; no elegimos 100 sin evidencia. Este promedio se calcula por escenario y no usa `y` ni los IDs de carta.

Después, una regresión logística de una sola variable aprende en train la pendiente y el intercepto que convierten ese score en probabilidad. No aprende pesos separados para premios, PV y banca. Esta fórmula es una propuesta basada en recursos del juego; hay que revisarla antes de interpretar los resultados como evidencia final.

In [ ]:
# Ejecuta B0 y B1 por escenario y guarda sus métricas y parámetros.
resultados, ajustes = [], []
columnas = ['game_id', 'y', 'stage', 'own_prizes', 'opp_prizes', 'own_active_max_hp', 'opp_active_max_hp', 'diff_total_hp', 'diff_bench_n']
columnas_parquet = [columna for columna in columnas if columna != 'stage']  # La etapa se deriva del turno al leer cada día.
for escenario in SCENARIOS:
    stream = ScenarioStream(DATA_DIR, escenario, tuple(columnas_parquet))  # Lee un día a la vez para controlar la RAM.
    prevalencia, hp_unit = b0_and_hp_unit(stream)
    beta_b1 = fit_b1_mapping(stream, hp_unit, prevalencia)

    partes_val = []
    for dia in stream:
        val = dia.loc[dia['split'] == 'val', ['game_id', 'y', 'stage']].copy()
        if len(val):
            mascara_val = dia['split'] == 'val'
            val['pred_b1'] = b1_probability(dia.loc[mascara_val], hp_unit, beta_b1)
            partes_val.append(val)
    val = pd.concat(partes_val, ignore_index=True)
    resultados += evaluar(val, np.full(len(val), prevalencia), escenario, 'B0')
    resultados += evaluar(val, val['pred_b1'].to_numpy(), escenario, 'B1')
    ajustes.append({'escenario': escenario, 'prevalencia_B0_train': prevalencia, 'hp_unit_train': hp_unit,
                    'intercepto_B1': beta_b1[0], 'pendiente_B1': beta_b1[1],
                    'partidas_val': val.game_id.nunique()})
    print(f'Escenario {escenario}: listo')
    del stream, partes_val, val, beta_b1
    gc.collect()  # Solicita liberar memoria antes de cargar el escenario siguiente.

metricas_baselines = pd.DataFrame(resultados)
ajustes_baselines = pd.DataFrame(ajustes)
display(metricas_baselines.round(4))
display(ajustes_baselines.round(4))
metricas_baselines.to_csv('/content/metricas_B0_B1.csv', index=False)
ajustes_baselines.to_csv('/content/ajustes_B0_B1.csv', index=False)
print('Resúmenes guardados en /content/metricas_B0_B1.csv y /content/ajustes_B0_B1.csv')


## B2 - Regresión logística regularizada

B2 usa las 62 características numéricas del modelo principal, excluyendo IDs de carta. Lee lotes de Parquet y entrena con **todas las partidas de train**, sin el límite anterior de 5,000. Primero aprende `StandardScaler` solo con fit y después lo fija; `SGDClassifier(loss='log_loss', penalty='l2')` aprende la regresión logística durante varias pasadas completas (épocas). Los pesos se calculan por partida completa aunque cruce lotes.

Se conservan los nueve valores de `C` y cinco pliegues por partida. Para reproducir el objetivo ponderado anterior se usa `alpha=1/(C*G)`, donde G es el número de partidas de fit, con pesos de SGD normalizados a media 1. El optimizador es distinto: revisar convergencia antes de interpretar resultados finales.

El piloto usa todos los datos de train de A para formar los cinco pliegues y ejecuta solo uno con `C=1`: cuatro quintas partes de las partidas ajustan el modelo y el resto se evalúa. `full` ejecuta todos los pliegues y escenarios; luego reajusta finalistas con todo train y evalúa val. Los lotes escalados y el AUC exacto usan disco local temporal. No se promedian AUC por lote. Se guardan CSV después de cada ajuste y modelos de las finalistas; copiar los resultados fuera del runtime antes de desconectarlo.

Al actualizar el repositorio, vuelve a abrir esta versión del cuaderno: `git pull` no actualiza las celdas de un cuaderno que ya está abierto en Colab. Para medir RAM limpiamente, inicia un runtime nuevo y ejecuta setup, configuración B2 y ejecución B2; no hace falta ejecutar B0/B1 primero.


In [ ]:
# Importado aquí para que esta configuración no dependa de una celda anterior.
import pyarrow.parquet as pq
# El piloto usa un pliegue de A; cambia a 'full' después de revisar tiempo y RAM.
B2_MODE = 'pilot'
# El tamaño del lote controla la RAM; ambos modos utilizan todo train disponible.
B2_BATCH_SIZE = 32768
B2_SEED = 2026
B2_MAX_EPOCHS = 20
B2_ETA0 = 0.05
B2_GRADIENT_TOL = 1e-3
B2_OUTPUT_DIR = Path('/content/b2_sgd') / B2_MODE
B2_OPTIONS = dict(max_iter=B2_MAX_EPOCHS, seed=B2_SEED, eta0=B2_ETA0,
                  tolerance=B2_GRADIENT_TOL, cache_dir='/content', output_dir=B2_OUTPUT_DIR)
parquet_schema = pq.ParquetFile(sorted(DATA_DIR.glob('estados_*.parquet'))[0]).schema.names
FEATURES_B2 = feature_columns(parquet_schema)  # Excluye metadatos y los dos IDs de carta.
print('B2 incremental | train: TODAS las partidas | características:', len(FEATURES_B2),
      '| modo:', B2_MODE, '| lote:', B2_BATCH_SIZE, '| épocas máximas:', B2_MAX_EPOCHS, '| C:', C_VALUES)


In [ ]:
import time, resource

if B2_MODE == 'pilot':
    t0 = time.perf_counter()
    stream = ScenarioStream(DATA_DIR, 'A', tuple(FEATURES_B2), batch_size=B2_BATCH_SIZE)
    b2_pilot = b2_group_cv(stream, FEATURES_B2, c_values=(1.0,), max_folds=1, **B2_OPTIONS)
    elapsed = time.perf_counter() - t0
    peak_gb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)
    display(b2_pilot.round(4))
    print(f'Piloto: {elapsed / 60:.1f} min; máximo de RAM del proceso: {peak_gb:.2f} GiB')
    del stream
elif B2_MODE == 'full':
    cv_parts, summary_parts, candidate_parts, val_parts = [], [], [], []
    for escenario in SCENARIOS:
        stream = ScenarioStream(DATA_DIR, escenario, tuple(FEATURES_B2), batch_size=B2_BATCH_SIZE)
        cv_folds = b2_group_cv(stream, FEATURES_B2, **B2_OPTIONS)
        cv_summary, candidates = summarize_b2_cv(cv_folds)
        cv_parts.append(cv_folds)
        summary_parts.append(cv_summary)
        candidate_parts.extend({'escenario': escenario, 'C': c} for c in candidates)
        val_parts.append(fit_b2_candidates_val(stream, FEATURES_B2, candidates, **B2_OPTIONS))
        print(f'B2 escenario {escenario}: listo; candidatas C={candidates}')
        del stream, cv_folds, cv_summary, candidates
        gc.collect()
    b2_cv_folds = pd.concat(cv_parts, ignore_index=True)
    b2_cv_summary = pd.concat(summary_parts, ignore_index=True)
    b2_candidates = pd.DataFrame(candidate_parts)
    b2_val_metrics = pd.concat(val_parts, ignore_index=True)
    display(b2_cv_summary.round(4))
    display(b2_candidates)
    display(b2_val_metrics.round(4))
    b2_cv_folds.to_csv('/content/b2_cv_folds.csv', index=False)
    b2_cv_summary.to_csv('/content/b2_cv_summary.csv', index=False)
    b2_val_metrics.to_csv('/content/b2_val_finalists.csv', index=False)
    print('B2 guardó reportes CSV en /content y avances/modelos en', B2_OUTPUT_DIR)
else:
    raise ValueError("B2_MODE debe ser 'pilot' o 'full'.")


## Leer los resultados

AUC más alto indica mejor ordenamiento; log loss y Brier más bajos indican mejores probabilidades. B2 muestra promedio y variación entre pliegues para cada `C`. `fraccion_train=1.0` confirma que el experimento incluye todas las partidas útiles de train. `n_partidas_fit` cuenta las que ajustaron ese modelo; en CV excluye las del pliegue retenido.

`convergencia=False` significa que el gradiente del objetivo de entrenamiento todavía supera la tolerancia. Revisar `gradiente_train`, `objetivo_train` y aumentar `B2_MAX_EPOCHS` antes de interpretar resultados finales; 20 épocas es un presupuesto inicial, no una garantía. Si el objetivo se vuelve inestable, reducir `B2_ETA0`. Se conserva la época de menor objetivo de train, sin usar val para detener el ajuste. `max_iter_alcanzado` solo indica que se agotó el presupuesto. El pico de RAM reportado es el máximo histórico del proceso, no solo de B2.

La demo por filas queda separada del flujo principal: usa LBFGS y una muestra pequeña, por lo que no debe compararse directamente con estos resultados SGD. `calib` y `test` siguen reservados.
